<a href="https://colab.research.google.com/github/ashishmanwani448-design/Data-analytics-assignments/blob/main/AdvancedSQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Advanced SQL

###Q1) What is a Common Table Expression (CTE), and how does it improve SQL query readability?

**Common Table Expressions (CTEs)** are temporary, named result sets that you can reference within a single SQL statement (SELECT, INSERT, UPDATE, or DELETE). They are defined using the `WITH` clause, providing a way to break down complex queries into simpler, more manageable, and readable logical units.

**How CTEs improve SQL query readability:**

1.  **Modularity and Organization:** CTEs allow you to define intermediate result sets separately. This makes complex queries easier to understand by breaking them into smaller, logically named blocks. Instead of one very long and nested query, you have several distinct parts.

2.  **Simplification of Complex Logic:** When dealing with multiple subqueries, joins, or recursive operations, CTEs can significantly simplify the overall query structure. Each CTE can encapsulate a specific piece of logic, which can then be referenced by subsequent CTEs or the final SELECT statement.

3.  **Readability and Maintainability:** By giving meaningful names to intermediate results, CTEs make the query's intent clearer. Anyone reading the query can easily follow the data transformation steps, improving both readability and maintainability for future modifications.

4.  **Avoidance of Repeated Code:** If an intermediate result set is needed multiple times within the same query, a CTE allows you to define it once and reference it multiple times. This avoids code repetition, making the query shorter and less prone to errors.

5.  **Support for Recursion:** CTEs are essential for performing recursive queries in SQL, allowing you to traverse hierarchical or graph-like data structures (e.g., organizational charts, bill of materials).

**Example Structure of a CTE:**

```sql
WITH
  CTE_Name_1 AS (
    -- SQL query that defines CTE_Name_1
    SELECT column1, column2
    FROM tableA
    WHERE condition1
  ),
  CTE_Name_2 AS (
    -- SQL query that defines CTE_Name_2, potentially referencing CTE_Name_1
    SELECT columnA, columnB
    FROM CTE_Name_1
    WHERE condition2
  )
SELECT final_column1, final_column2
FROM CTE_Name_2
WHERE final_condition;
```

### Q2. Why are some views updatable while others are read-only? Explain with an example.

In SQL, views can be either **updatable** (meaning you can use `INSERT`, `UPDATE`, or `DELETE` statements on them, which in turn modifies the underlying base table) or **read-only** (meaning you can only query data from them). The ability to update a view depends on how the view is defined and whether the database system can unambiguously map the changes back to the base tables.

#### Why Views Become Read-Only (Non-Updatable):

Views become read-only when the database system cannot determine how to correctly or unambiguously modify the data in the underlying base table(s) based on the changes made to the view. Common reasons include:

1.  **Aggregate Functions:** If the view includes aggregate functions (`SUM()`, `AVG()`, `COUNT()`, `MIN()`, `MAX()`), the database cannot logically reverse the aggregation to update individual base table rows.

2.  **`GROUP BY` Clause:** Similar to aggregate functions, grouping data makes it impossible to identify which specific rows in the base table should be modified.

3.  **`DISTINCT` Keyword:** If the view uses `DISTINCT` to remove duplicate rows, the database cannot determine which of the original duplicate rows to update.

4.  **`JOIN` Operations (Complex Joins):** Views created by joining multiple tables are often read-only, especially if the update affects columns from more than one table or if the join is not a simple one-to-one relationship. The database might not know which table to update or how to maintain data integrity across joined tables.

5.  **`UNION`, `UNION ALL`, `INTERSECT`, `EXCEPT`:** Views based on these set operators combine results from multiple queries, making them non-updatable because changes cannot be unambiguously mapped back to the original source.

6.  **Subqueries in `SELECT` List:** If a subquery is used in the `SELECT` list of the view definition, it usually makes the view non-updatable.

7.  **Derived Columns (Calculated Columns):** If the view's `SELECT` list includes columns that are derived from expressions or calculations (e.g., `columnA + columnB AS new_column`), these derived columns cannot be updated directly, and often make the entire view non-updatable.

8.  **Certain `WHERE` Clause Conditions:** In some database systems, a `WHERE` clause that references non-key-preserved tables in a join might make the view read-only.

#### Why Views are Updatable:

Views are generally updatable if they are based on a single base table and do not contain any of the constructs listed above that prevent updates. Specifically:

*   **Single Base Table:** The view must select columns from only one base table.
*   **No Aggregate Functions, `GROUP BY`, `DISTINCT`:** As mentioned, these make views read-only.
*   **No Derived Columns:** All columns in the view must map directly to columns in the base table.
*   **Key-Preserved Table:** In multi-table views (which are rarely updatable), if you manage to update a view that has a join, the columns being updated must belong to a 'key-preserved table' in the view, meaning that every key of the base table can also be identified as a key in the view.

#### Example:

Let's consider a simple `Products` table:

```sql
CREATE TABLE Products (
    ProductID INT PRIMARY KEY,
    ProductName VARCHAR(100),
    Price DECIMAL(10, 2),
    QuantityInStock INT
);

INSERT INTO Products (ProductID, ProductName, Price, QuantityInStock) VALUES
(1, 'Laptop', 1200.00, 50),
(2, 'Mouse', 25.00, 200),
(3, 'Keyboard', 75.00, 100);
```

**1. Updatable View:**

This view is based on a single table and contains no complex operations.

```sql
CREATE VIEW CurrentStock AS
SELECT ProductID, ProductName, QuantityInStock
FROM Products
WHERE QuantityInStock > 0;
```

You **can** update this view:

```sql
UPDATE CurrentStock
SET QuantityInStock = 45
WHERE ProductID = 1;

-- The underlying Products table will be updated.
SELECT * FROM Products WHERE ProductID = 1;
-- Result will show QuantityInStock = 45
```

**2. Read-Only View:**

This view includes an aggregate function (`SUM()`) and a `GROUP BY` clause.

```sql
CREATE VIEW TotalStockValue AS
SELECT SUM(Price * QuantityInStock) AS TotalValue
FROM Products;
```

You **cannot** update this view because the database wouldn't know how to translate a change in `TotalValue` back to individual `Price` or `QuantityInStock` values for specific products.

Attempting to update it would result in an error:

```sql
-- This would result in an error like 'SQLSTATE[HY000]: General error: Cannot modify an aggregate view.'
-- UPDATE TotalStockValue
-- SET TotalValue = 100000.00;
```

Similarly, a view with a `JOIN` could be read-only if the update logic is ambiguous:

```sql
CREATE TABLE Orders (
    OrderID INT PRIMARY KEY,
    ProductID INT,
    OrderQuantity INT,
    OrderDate DATE
);

INSERT INTO Orders (OrderID, ProductID, OrderQuantity, OrderDate) VALUES
(101, 1, 2, '2023-10-26'),
(102, 3, 1, '2023-10-26');

CREATE VIEW ProductOrderDetails AS
SELECT p.ProductName, o.OrderQuantity, o.OrderDate
FROM Products p
JOIN Orders o ON p.ProductID = o.ProductID;
```

This `ProductOrderDetails` view would typically be read-only because modifying `ProductName` or `OrderQuantity` through the view would involve updating different underlying tables (`Products` and `Orders`), which can lead to ambiguity or data integrity issues that the database prevents.

### Q3. What advantages do stored procedures offer compared to writing raw SQL queries repeatedly?

**Stored procedures** are a set of SQL statements that are compiled and stored on the database server. Once created, they can be executed by applications or users, offering several significant advantages over repeatedly writing and executing raw SQL queries:

1.  **Improved Performance:**
    *   **Pre-compilation:** When a stored procedure is first created, it is compiled and stored in its compiled form. This means the database doesn't need to parse, optimize, and compile the query plan every time it's executed, leading to faster execution times.
    *   **Reduced Network Traffic:** Instead of sending multiple raw SQL statements over the network, only a single call to the stored procedure is made, which reduces network overhead.

2.  **Enhanced Security:**
    *   **Access Control:** Users can be granted permissions to execute a stored procedure without having direct permissions on the underlying tables. This allows developers to control how data is accessed and modified, providing a layer of abstraction and security.
    *   **SQL Injection Prevention:** Stored procedures can be designed to accept parameters, which helps in preventing SQL injection attacks, as the parameters are treated as data rather than executable code.

3.  **Code Reusability and Modularity:**
    *   **Write Once, Use Many Times:** A stored procedure can be written once and called from multiple applications or parts of an application. This promotes code reuse and reduces redundancy.
    *   **Modularity:** Complex database operations can be encapsulated into smaller, manageable stored procedures, improving the organization and maintainability of the database code.

4.  **Simplified Maintenance:**
    *   **Centralized Logic:** If the underlying database schema changes, only the stored procedure might need to be updated, rather than changing numerous raw SQL queries scattered across various applications.
    *   **Easier Debugging:** Many database systems provide tools for debugging stored procedures, which can simplify the process of identifying and fixing issues in complex logic.

5.  **Transactional Control:**
    *   Stored procedures can incorporate `BEGIN TRANSACTION`, `COMMIT`, and `ROLLBACK` statements, ensuring that a series of operations are treated as a single, atomic unit. This is crucial for maintaining data integrity.

6.  **Abstraction:**
    *   Stored procedures provide an abstraction layer over the database. Application developers don't need to know the intricate details of the database schema; they just need to know how to call the relevant stored procedures.

**In summary,** while raw SQL queries are flexible for ad-hoc data retrieval, stored procedures offer significant benefits in terms of performance, security, reusability, and maintainability for recurring and complex database operations.

### Q4. What is the purpose of triggers in a database? Mention one use case where a trigger is essential.

**Triggers** are special stored procedures that are automatically executed or 'fired' when a specific event occurs in the database. These events can be data modification statements (`INSERT`, `UPDATE`, `DELETE`) on a table or view, or even database operations like `LOGON`, `LOGOFF`, or `SERVER ERROR` (though statement-level triggers are more common for data manipulation).

**The primary purposes of triggers include:**

1.  **Enforcing Complex Business Rules/Integrity Constraints:** Triggers can enforce complex business rules that cannot be handled by standard declarative constraints (like `PRIMARY KEY`, `FOREIGN KEY`, `CHECK`). For example, ensuring that a product's price cannot be decreased by more than 10% in a single update.

2.  **Auditing and Logging:** Triggers are excellent for keeping an audit trail of changes made to data. They can record who changed what, when, and from which application, into a separate audit table.

3.  **Data Synchronization:** Triggers can be used to automatically update related data in other tables or systems when a change occurs in a primary table. This helps maintain data consistency across different parts of the database or integrated systems.

4.  **Deriving Column Values:** They can automatically calculate and populate values for derived columns based on changes in other columns within the same or different tables.

5.  **Preventing Invalid Transactions:** Triggers can check conditions before or after a data modification event and, if conditions are not met, can roll back the transaction, preventing invalid data from entering the database.

#### Essential Use Case: Maintaining an Audit Log

One use case where a trigger is **essential** is **maintaining an audit log for sensitive data changes**. For example, in a financial system, it's crucial to record every modification made to a customer's account balance, personal information, or transaction history.

**Why it's essential:**

*   **Accountability:** An audit log provides a record of who made changes, which is vital for accountability and security.
*   **Compliance:** Many regulatory requirements (e.g., GDPR, HIPAA, financial regulations) mandate strict auditing of data access and modification.
*   **Troubleshooting and Recovery:** If data corruption or an erroneous transaction occurs, the audit log can help trace the sequence of events, identify the cause, and potentially aid in data recovery.
*   **Non-Repudiation:** It ensures that a user cannot deny making a particular change, as their actions are automatically recorded by the database itself, outside the application's direct control.

**Example (Conceptual):**

Imagine a `Customers` table with sensitive information. A `BEFORE UPDATE` or `AFTER UPDATE` trigger could be set up to record every change:

```sql
-- Assuming an Audit_Log table exists
CREATE TABLE Audit_Log (
    LogID INT IDENTITY(1,1) PRIMARY KEY,
    TableName VARCHAR(100),
    RecordID INT,
    FieldName VARCHAR(100),
    OldValue VARCHAR(MAX),
    NewValue VARCHAR(MAX),
    ChangedBy VARCHAR(100),
    ChangeDate DATETIME DEFAULT GETDATE()
);

-- Example Trigger for SQL Server (syntax varies by RDBMS)
CREATE TRIGGER trg_Customers_Audit
ON Customers
AFTER UPDATE
AS
BEGIN
    INSERT INTO Audit_Log (TableName, RecordID, FieldName, OldValue, NewValue, ChangedBy)
    SELECT
        'Customers',
        i.CustomerID,
        COLUMN_NAME,
        d.COLUMN_NAME,
        i.COLUMN_NAME,
        SYSTEM_USER -- Or a more sophisticated way to get the user
    FROM
        inserted i
    INNER JOIN
        deleted d ON i.CustomerID = d.CustomerID
    WHERE
        -- Compare values for relevant columns and log only if they changed
        i.FirstName <> d.FirstName OR
        i.LastName <> d.LastName OR
        i.Email <> d.Email;

    -- More robust triggers would iterate through all relevant columns and insert multiple rows per update
END;
```

This trigger ensures that no matter how the `Customers` table is updated (whether through an application, direct SQL query, or stored procedure), the changes are automatically logged, providing a consistent and reliable audit trail.

### Q5. Explain the need for data modelling and normalization when designing a database.

When designing a database, **data modeling** and **normalization** are crucial steps that ensure the database is efficient, reliable, maintainable, and scalable. They work hand-in-hand to create a robust database structure.

#### 1. Data Modeling

**Data modeling** is the process of creating a visual representation or blueprint of an organization's data, showing how data relates to other data. It involves identifying the types of data that will be stored, the relationships among these data types, and the rules governing their interactions.

**Need for Data Modeling:**

*   **Understanding Business Requirements:** It helps in clearly understanding and documenting the business processes and requirements, translating them into a database structure.
*   **Clear Communication:** Provides a common language and visual representation for business users, developers, and database administrators to discuss and validate the database design.
*   **Reduced Errors and Redundancy:** By carefully designing the data structure, data modeling helps identify and eliminate potential inconsistencies, redundancies, and errors early in the development cycle.
*   **Improved Data Quality:** Ensures that data is stored accurately and consistently, leading to higher data quality.
*   **Better Database Design:** Guides the creation of an efficient and organized database schema, making it easier to manage, query, and update data.
*   **Foundation for Normalization:** A well-defined data model (often an Entity-Relationship Diagram or ERD) provides the basis for applying normalization techniques.

There are different levels of data modeling:
*   **Conceptual Data Model:** High-level, abstract view of data, independent of technology.
*   **Logical Data Model:** More detailed, defines entities, attributes, and relationships, but still independent of specific DBMS.
*   **Physical Data Model:** Specifies how data will be stored in a specific database management system (DBMS), including data types, indexes, and constraints.

#### 2. Normalization

**Normalization** is a systematic process of organizing the columns and tables of a relational database to minimize data redundancy and improve data integrity. It involves breaking down large tables into smaller, related tables and defining relationships between them.

**Need for Normalization:**

*   **Reduce Data Redundancy:** Eliminates duplicate data, which saves storage space and prevents inconsistencies that can arise when the same data is stored in multiple places and only some instances are updated.
*   **Improve Data Integrity:** Ensures that data dependencies make sense, meaning that data is stored in the correct table. This prevents update, insert, and delete anomalies:
    *   **Insert Anomaly:** Cannot add a new tuple to a table without adding values for other attributes.
    *   **Update Anomaly:** Updating one data item requires updating multiple records, and if one is missed, data becomes inconsistent.
    *   **Delete Anomaly:** Deleting a record might unintentionally remove other valuable information.
*   **Enhance Data Consistency:** By ensuring that each piece of data is stored only once, normalization helps maintain consistency across the database.
*   **Simplify Queries and Maintenance:** Smaller, well-structured tables are easier to understand, query, and maintain. Changes to the schema or data are more straightforward.
*   **Improve Performance (in some cases):** While often associated with increased joins, well-normalized tables can improve query performance by reducing the amount of data that needs to be scanned and by making indexes more effective for specific queries.

**Normal Forms:** Normalization typically progresses through a series of forms, with each form addressing specific types of data redundancy and integrity issues:

*   **First Normal Form (1NF):** Each column contains atomic (indivisible) values, and there are no repeating groups of columns.
*   **Second Normal Form (2NF):** Is in 1NF and all non-key attributes are fully functionally dependent on the primary key (no partial dependencies).
*   **Third Normal Form (3NF):** Is in 2NF and has no transitive functional dependencies (i.e., non-key attributes are not dependent on other non-key attributes).
*   **Boyce-Codd Normal Form (BCNF):** A stricter version of 3NF, where every determinant is a candidate key.

In practice, databases are often normalized up to 3NF or BCNF. Sometimes, **denormalization** (intentionally introducing redundancy) is used to improve query performance for specific use cases, but this is a conscious decision made after considering the trade-offs in data integrity and maintenance.

Q6. Write a CTE to calculate the total revenue for each product
 (Revenues = Price × Quantity), and return only products where  revenue > 3000.

 Ans:

 ***Query:***

    WITH revenue AS (
    SELECT
        s.Product1D,
        p.ProductName,
        p.Price,
        s.Quantity,
        (p.Price * s.Quantity) AS total_revenue
    FROM Sales s
    JOIN Products p
        ON s.Product1D = p.Product1D
    WHERE (p.Price * s.Quantity) > 3000
    )
    SELECT *
    FROM revenue



 ***Result:***

Q7) Ceate a view named vw_CategorySummary that shows:
 Category, TotalProducts, AveragePrice.

  Ans:

 ***Query:***

    CREATE VIEW vw_CategorySummary AS
    SELECT
    Category,
    COUNT(Product1D) AS TotalProducts,
    AVG(Price) AS AveragePrice
    FROM Products
    GROUP BY Category;
    select * from vw_CategorySummary;


 ***Result:***

Q8) Create an updatable view containing ProductID, ProductName, and Price.
 Then update the price of ProductID = 1 using the view.
 Ans:


 ***Query:***

    Create view updatabale_view as
    select Product1D, ProductName, Price from Products ;

    Update updatabale_view
    set price = 10000
    where product1D=1;

    Select * from updatabale_view;


 ***Result:***

Q9. Create a stored procedure that accepts a category name and returns all products belonging to that
category.

Ans:

 ***Query:***

    delimiter $$
    create procedure Categorynames(in Name_Category Varchar(50))
    begin
    select ProductName from products
    where Name_category = Category ;
    end$$

    Delimiter ;

    call Categoryname('Electronics');


 ***Result:***

Q10. Create an AFTER DELETE trigger on the Products table that archives deleted product rows into a new table ProductArcheive. The archive should store ProductID, ProductName, Category, Price, and DeletedAt
timestamp.

Ans:

***Query:***

    1) Create table deleted_Archeive(
    ProductID int primary key ,
    ProductName varchar(50),
    Category varchar(50),
    Price Decimal(10,2),
    DeletedAt time
    );


    2) Delimiter $$
    create trigger Deleted_data1
    after delete on Products
    for each row
    begin
    insert into deleted_Archeive
    Values(ProductID, ProductName, Category, Price, now());
    End$$

    Delimiter ;

    3)  delete from Products
        where product1D = 2;

    4) Select * Deleted_Archeive


***Result:***